# 03 - Integracao e Feature Engineering

Objetivo deste notebook:
- integrar as bases tratadas por `codigo_municipio + ano`
- criar a base analitica principal do projeto
- gerar features para BI, Data Warehouse e Machine Learning
- salvar a base final em `datasets/tratados/base_analitica_municipio_ano.csv`

Granularidade da base final:

`1 linha = 1 capital brasileira + 1 ano`


## 1. Imports e caminhos

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PASTA_TRATADOS = Path('../datasets/tratados')
CAMINHO_BASE_ANALITICA = PASTA_TRATADOS / 'base_analitica_municipio_ano.csv'


## 2. Funcoes auxiliares

In [ ]:
def validar_chave(df: pd.DataFrame, chave: list[str], nome: str) -> None:
    duplicatas = df.duplicated(subset=chave).sum()
    print(f'{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas')
    print(f'Duplicatas na chave {chave}: {duplicatas}')
    if duplicatas:
        display(df[df.duplicated(subset=chave, keep=False)].sort_values(chave).head(20))


def adicionar_taxa_100k(df: pd.DataFrame, coluna_numerador: str, coluna_saida: str) -> pd.DataFrame:
    resultado = df.copy()
    resultado[coluna_saida] = (resultado[coluna_numerador] / resultado['populacao_total']) * 100000
    return resultado


def mostrar_nulos(df: pd.DataFrame) -> None:
    display(df.isna().sum().sort_values(ascending=False).to_frame('qtd_nulos'))


## 3. Carregar bases tratadas

In [ ]:
df_idh = pd.read_csv(PASTA_TRATADOS / 'idh_municipio_2010_tratado.csv')
df_crimes = pd.read_csv(PASTA_TRATADOS / 'crimes_municipio_ano_tratado.csv')
df_populacao = pd.read_csv(PASTA_TRATADOS / 'populacao_municipio_ano_tratado.csv')
df_educacao = pd.read_csv(PASTA_TRATADOS / 'educacao_ideb_uf_ano_total_em_tratado.csv')

bases = {
    'idh': df_idh,
    'crimes': df_crimes,
    'populacao': df_populacao,
    'educacao': df_educacao,
}

for nome, df in bases.items():
    print(nome, df.shape)
    display(df.head())


## 4. Validar granularidade das entradas

In [ ]:
validar_chave(df_idh, ['codigo_municipio'], 'IDH')
validar_chave(df_crimes, ['codigo_municipio', 'ano'], 'Crimes')
validar_chave(df_populacao, ['codigo_municipio', 'ano'], 'Populacao')
validar_chave(df_educacao, ['codigo_uf_ibge', 'ano'], 'Educacao IDEB UF ano')


## 5. Integrar bases

In [ ]:
df_base = df_populacao.copy()

colunas_crimes_para_remover = [
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
]
df_crimes_metricas = df_crimes.drop(columns=[coluna for coluna in colunas_crimes_para_remover if coluna in df_crimes.columns])

df_base = df_base.merge(
    df_crimes_metricas,
    how='left',
    on=['codigo_municipio', 'ano'],
    validate='one_to_one'
)

df_idh_metricas = df_idh[[
    'codigo_municipio',
    'ano_referencia_idhm',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
]].copy()

df_base = df_base.merge(
    df_idh_metricas,
    how='left',
    on='codigo_municipio',
    validate='many_to_one'
)

df_educacao_metricas = df_educacao[[
    'codigo_uf_ibge',
    'ano',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
]].copy()

df_base = df_base.merge(
    df_educacao_metricas,
    how='left',
    on=['codigo_uf_ibge', 'ano'],
    validate='many_to_one'
)

df_base.shape


## 6. Tratar nulos apos integracao

In [ ]:
mostrar_nulos(df_base)


In [ ]:
colunas_quantidade = [coluna for coluna in df_base.columns if coluna.startswith('quantidade_')]
colunas_proporcao = [coluna for coluna in df_base.columns if coluna.startswith('proporcao_')]

df_base[colunas_quantidade] = df_base[colunas_quantidade].fillna(0)
df_base[colunas_proporcao] = df_base[colunas_proporcao].fillna(0)

mostrar_nulos(df_base)


## 7. Criar indicadores criminais agregados

In [ ]:
componentes_crime_total = [
    'quantidade_mortes_violentas_intencionais',
    'quantidade_feminicidio',
    'quantidade_estupro',
    'quantidade_furto_veiculos',
    'quantidade_roubo_veiculos',
    'quantidade_latrocinio',
    'quantidade_lesao_corporal_morte',
]
componentes_crime_total = [coluna for coluna in componentes_crime_total if coluna in df_base.columns]

df_base['crimes_total_indicadores'] = df_base[componentes_crime_total].sum(axis=1)

df_base['homicidios_dolosos'] = df_base.get('quantidade_homicidio_doloso', 0)
df_base['mortes_violentas_intencionais'] = df_base.get('quantidade_mortes_violentas_intencionais', 0)
df_base['feminicidios'] = df_base.get('quantidade_feminicidio', 0)
df_base['estupros'] = df_base.get('quantidade_estupro', 0)
df_base['furto_veiculos'] = df_base.get('quantidade_furto_veiculos', 0)

df_base[[
    'codigo_municipio',
    'ano',
    'crimes_total_indicadores',
    'mortes_violentas_intencionais',
    'homicidios_dolosos',
    'feminicidios',
    'estupros',
    'furto_veiculos',
]].head()


## 8. Criar taxas por 100 mil habitantes

In [ ]:
taxas = {
    'crimes_total_indicadores': 'taxa_crimes_100k',
    'mortes_violentas_intencionais': 'taxa_mortes_violentas_100k',
    'homicidios_dolosos': 'taxa_homicidios_100k',
    'feminicidios': 'taxa_feminicidios_100k',
    'estupros': 'taxa_estupros_100k',
    'furto_veiculos': 'taxa_furto_veiculos_100k',
}

for numerador, saida in taxas.items():
    df_base = adicionar_taxa_100k(df_base, numerador, saida)

df_base[list(taxas.values())].describe().transpose()


## 9. Criar features temporais

In [ ]:
df_base = df_base.sort_values(['codigo_municipio', 'ano']).copy()

df_base['lag_1_taxa_crimes_100k'] = df_base.groupby('codigo_municipio')['taxa_crimes_100k'].shift(1)
df_base['lag_1_taxa_mortes_violentas_100k'] = df_base.groupby('codigo_municipio')['taxa_mortes_violentas_100k'].shift(1)
df_base['variacao_taxa_crimes_100k_pct'] = df_base.groupby('codigo_municipio')['taxa_crimes_100k'].pct_change() * 100

df_base[[
    'codigo_municipio',
    'nome_municipio',
    'ano',
    'taxa_crimes_100k',
    'lag_1_taxa_crimes_100k',
    'variacao_taxa_crimes_100k_pct',
]].head(12)


## 10. Criar indice simples de risco

In [ ]:
componentes_risco = [
    'taxa_mortes_violentas_100k',
    'taxa_homicidios_100k',
    'taxa_feminicidios_100k',
    'taxa_estupros_100k',
]

for coluna in componentes_risco:
    minimo = df_base[coluna].min()
    maximo = df_base[coluna].max()
    if maximo == minimo:
        df_base[f'{coluna}_norm'] = 0
    else:
        df_base[f'{coluna}_norm'] = (df_base[coluna] - minimo) / (maximo - minimo)

df_base['risco_indice'] = df_base[[f'{coluna}_norm' for coluna in componentes_risco]].mean(axis=1)

df_base[['nome_municipio', 'ano', 'risco_indice']].sort_values('risco_indice', ascending=False).head(10)


## 11. Selecionar colunas finais da base analitica

In [ ]:
colunas_finais = [
    'codigo_municipio',
    'codigo_uf_ibge',
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
    'ano',
    'populacao_total',
    'populacao_crescimento_pct',
    'ano_referencia_idhm',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
    'crimes_total_indicadores',
    'mortes_violentas_intencionais',
    'homicidios_dolosos',
    'feminicidios',
    'estupros',
    'furto_veiculos',
    'taxa_crimes_100k',
    'taxa_mortes_violentas_100k',
    'taxa_homicidios_100k',
    'taxa_feminicidios_100k',
    'taxa_estupros_100k',
    'taxa_furto_veiculos_100k',
    'lag_1_taxa_crimes_100k',
    'lag_1_taxa_mortes_violentas_100k',
    'variacao_taxa_crimes_100k_pct',
    'risco_indice',
]

df_base_analitica = df_base[colunas_finais].copy()

df_base_analitica.head()


## 12. Validacoes finais

In [ ]:
validar_chave(df_base_analitica, ['codigo_municipio', 'ano'], 'Base analitica municipio ano')
print('Anos:', sorted(df_base_analitica['ano'].unique()))
print('Municipios:', df_base_analitica['codigo_municipio'].nunique())
mostrar_nulos(df_base_analitica)


## 13. Salvar base analitica

In [ ]:
df_base_analitica.to_csv(CAMINHO_BASE_ANALITICA, index=False)
CAMINHO_BASE_ANALITICA


# Resultado da etapa

Arquivo gerado:

`datasets/tratados/base_analitica_municipio_ano.csv`

Essa base sera usada por:
- notebook 04 de modelagem
- notebook 05 de carga no PostgreSQL / DW
- Data Mart e Metabase
